# Experiment 6.0 — Multi-timescale phase-aware evidence

Analysis-only notebook. All training, checkpoint selection, test evaluation, activity extraction, and aggregation are completed by the Exp6.0 CPU pipeline before this notebook is opened.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

REPO_ROOT = Path('..').resolve()
ROOT = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_6_0_multiscale_phase_evidence' / 'single_hidden_multiscale_phase_evidence_v1'
runs = pd.read_csv(ROOT / 'runs.csv')
summary = pd.read_csv(ROOT / 'summary.csv')
history_index = pd.read_csv(ROOT / 'history_index.csv')
baseline = json.loads((ROOT / 'baseline_fixed250_linear.json').read_text())
manifest = json.loads((ROOT / 'manifest.json').read_text())
sample = json.loads((ROOT / 'visualization_sample.json').read_text())
print('runs:', len(runs))
print('visualization sample:', sample)


## Run-level results and Fixed250 reference

In [ ]:
display(runs.sort_values(['objective', 'mem_shift', 'seed']))
display(summary)
baseline_metrics = baseline['baseline']['metrics']
pd.DataFrame(baseline_metrics).T


## Test Balanced Accuracy vs hidden membrane shift

In [ ]:
labels = {
    'whole_count': 'WholeCount',
    'whole_count_hce': 'WholeCount + HCE',
    'whole_count_contextual_gain': 'WholeCount + Contextual Gain',
}
fig, ax = plt.subplots(figsize=(8, 5))
for objective, label in labels.items():
    subset = runs[runs.objective == objective]
    stats = subset.groupby('mem_shift').test_ba.agg(['mean', 'std']).reindex([1, 2, 3])
    ax.errorbar(stats.index, stats['mean'], yerr=stats['std'].fillna(0), marker='o', capsize=4, label=label)
reference = float(baseline_metrics['test']['balanced_accuracy'])
ax.axhline(reference, linestyle='--', label=f'Raw Fixed250 + Linear ({reference:.3f})')
ax.set_xticks([1, 2, 3])
ax.set_xlabel('Hidden membrane shift')
ax.set_ylabel('Test Balanced Accuracy')
ax.set_ylim(0, 1)
ax.legend()
plt.show()


## Validation Balanced Accuracy vs hidden membrane shift

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for objective, label in labels.items():
    subset = runs[runs.objective == objective]
    stats = subset.groupby('mem_shift').val_ba.agg(['mean', 'std']).reindex([1, 2, 3])
    ax.errorbar(stats.index, stats['mean'], yerr=stats['std'].fillna(0), marker='o', capsize=4, label=label)
ax.set_xticks([1, 2, 3])
ax.set_xlabel('Hidden membrane shift')
ax.set_ylabel('Validation Balanced Accuracy')
ax.set_ylim(0, 1)
ax.legend()
plt.show()


## Hidden s4/s5/s6 firing diagnostics

In [ ]:
fr_cols = ['hidden_s4_fr_valid', 'hidden_s5_fr_valid', 'hidden_s6_fr_valid', 'hidden_s4_fr_tail', 'hidden_s5_fr_tail', 'hidden_s6_fr_tail']
display(runs.groupby(['objective', 'mem_shift'])[fr_cols].agg(['mean', 'std']))


## Example learning curve and spike raster

All runs use the same deterministic validation segment; the cells below only display finalized PNGs.

In [ ]:
row = history_index.sort_values(['objective', 'mem_shift', 'seed']).iloc[0]
display(Image(filename=str(REPO_ROOT / row.learning_curve_png)))
display(Image(filename=str(REPO_ROOT / row.raster_png)))


## HCE original-order vs reversed-order raster

In [ ]:
hce = runs[runs.objective == 'whole_count_hce'].sort_values(['mem_shift', 'seed']).iloc[0]
key = f"whole_count_hce__mem{int(hce.mem_shift)}__seed{int(hce.seed)}"
display(Image(filename=str(ROOT / 'rasters' / f'{key}.png')))
display(Image(filename=str(ROOT / 'rasters' / f'{key}__reversed.png')))


## Interpretation checklist

1. Does HCE improve BA relative to WholeCount at matched membrane shift and seed?
2. Does Contextual Gain improve BA and produce positive middle/long gain diagnostics?
3. Does increasing hidden membrane shift improve useful temporal context or mainly change firing/tail persistence?
4. How close does the best causal SNN get to Raw Fixed250 + Linear?
5. Do the original/reversed HCE rasters show meaningful changes especially in s5/s6 activity?